In [0]:
import json
import re
import os
from databricks.sdk import WorkspaceClient

# 1. Initialize the client
w = WorkspaceClient()

print("Starting Genie Space export...")
workspace_folder_path = "/Workspace/Users/ibhalla@infomedia.com.au/Notebooks/Genie Worspaces/"

# 'exist_ok=True' means it won't error if the folder already exists
os.makedirs(workspace_folder_path, exist_ok=True)
print(f"Ensuring output directory exists: {workspace_folder_path}\n")
# ------------------------------------------------------------------

# 2. Fetch the response and loop through the spaces list
response = w.genie.list_spaces()

# Check if spaces exist in the response
if hasattr(response, 'spaces') and response.spaces:
    for space_summary in response.spaces:
        try:
            # Fetch the full details
            space = w.genie.get_space(
                space_id=space_summary.space_id, 
                include_serialized_space=True
            )
            
            # Clean up the space name to make it a valid filename 
            space_name = space_summary.title or space_summary.space_id
            safe_name = re.sub(r'[^\w\-_\.]', '_', space_name)
            
            # Construct the full workspace path ---
            filename_only = f"{safe_name}.json"
            full_filepath = os.path.join(workspace_folder_path, filename_only)
            # ---------------------------------------------------
            
            # Parse and save the JSON
            if space.serialized_space:
                config_data = json.loads(space.serialized_space)
                
                # Use the full filepath to write directly to the workspace ---
                with open(full_filepath, "w") as f:
                    json.dump(config_data, f, indent=2)
                
                # Show the full workspace path in the output for confirmation
                print(f"✅ Exported: {space_name} -> {full_filepath}")
            else:
                print(f"⚠️ Skipped {space_name}: No configuration found.")
                
        except Exception as e:
            print(f"❌ Error exporting {space_summary.title}: {e}")
else:
    print("No Genie spaces found to export.")